In [1]:
import re
import random
import numpy as np
from typing import Dict, List

from bsbolt.Utils.CGmapIterator import OpenCGmap
from bsbolt.Simulate.SimulationOutput import SimulationOutput

In [2]:
class SetCytosineMethylation:

    def __init__(self, reference_file: str = None, sim_output: str = None, 
                 meth_ref: str = None, cgmap: str = None, beta_param: dict = None,
                 collect_ch_sites: bool = True, overwrite_db: bool = False):
        
        self.reference_file = reference_file
        self.reference_dict = SeqIO.to_dict(SeqIO.parse(reference_file, "fasta"))
        
        self.sim_output = sim_output
        self.beta_param = beta_param
        self.sim_meth_db: SimulationOutput = None
        
        self.overwrite_db = overwrite_db
        self.collect_ch_sites = collect_ch_sites # currently use all of the Cs
        self.initialize_methylation_reference(methylation_reference, cgmap)
    
    
    def initialize_methylation_reference(self, methylation_reference, cgmap):
        """If profile exists then assume already run"""
        self.profile_dict= dict()
        self.sim_meth_db = SimulationOutput(sim_output=self.sim_output)
        self.sim_meth_db.generate_sim_directory()
        
        
        # initialize data object using fasta sequence
        for contig_id in self.reference_dict.keys():
            contig_profile = dict()
            contig_seq = self.reference_dict[contig_id].seq
            
            for c_pos in re.finditer("C", str(self.reference_dict[contig_id].seq)):
                context = 1 if str(contig_seq[c_pos:(c_pos+2)]) == 'CG' else 0
                contig_profile[c_pos] = np.array([0, context, np.NaN], dtype=float16)
            for g_pos in re.finditer("G", str(self.reference_dict[contig_id].seq)):
                context = 1 if str(contig_seq[c_pos:(c_pos+2)]) == 'GC' else 0
                contig_profile[g_pos] = np.array([1, context, np.NaN], dtype=float16)
            self.profile_dict[contig_id] = contig_profile
        
        
        # intialize methylation value based on provided profile
        if methylation_reference:
            self.sim_meth_db_from_dist(beta_param) # TODO: load the existing meth_db.pkl file
        elif cgmap:
            self.sim_meth_db_from_cgmap(cgmap)
        else:
            self.sim_meth_db_from_dist(beta_param)

    def sim_meth_db_from_dist(beta_param):
        pass
    
    def sim_meth_db_from_cgmap(self, cgmap: str):
        for line in OpenCGmap(cgmap):
            chrom, nucleotide, pos, context, methlevel = line[0], line[1], int(line[2]) - 1, line[3], float(line[5])
            context = 1 if context == 'CG' else 0
            nucleotide = 1 if nucleotide == 'G' else 0
            self.profile_dict[chrom][pos] = np.array([nucleotide, context, methlevel], dtype=float16)
        
        for contig_id in self.profile_dict.keys():
            for meth_pos, meth_profile in self.profile_dict[contig_id].items():
                if np.isnan(meth_profile[2]):
                    self.profile_dict[contig_id][meth_pos] = self.pick_cytosine_methylation(meth_profile[1])
                    
        for contig_id, contig_profile in meth_keys.items():
            self.sim_meth_db.output_contig(contig_id, contig_profile, values=True)

    @property
    def pick_cytosine_methylation(self, context):
        """Sample from Cytosine distribution, 1 for CG and 0 for CH"""
        if context:
            return np.random.beta(self.beta_param['CG'][0], self.beta_param['CG'][1], size =1)[0]
        return np.random.beta(self.beta_param['CHG'][0], self.beta_param['CHG'][1], size =1)[0]
        
        
    def get_contig_methylation(self, contig):
        contig_profile = self.sim_meth_db.load_contig(contig, values=True)
        if contig_profile:
            return contig_profile


    def set_variant_methylation(self, sim_data, contig_profile, current_contig):
        """Variants are always random, so set random methylation"""
        ref_seq = self.reference[current_contig]
        for pos, variant_info in sim_data.items():
            # don't set methylation for last base
            if pos + 2 > len(ref_seq):
                continue
            if variant_info['indel'] == -1:
                continue
            elif variant_info['indel'] == 1:
                insert_context = f'{ref_seq[pos-1]}{variant_info["alt"]}{ref_seq[pos]}'
                for insert_pos, base in enumerate(variant_info['alt']):
                    if base == 'C':
                        nucleotide = 0
                        context = 1 if insert_context[insert_pos+1] == "G" else 0
                    if base == "G":
                        nucleotide = 1
                        context = 1 if insert_context[insert_pos+1] == "C" else 0
                    
                    meth_profile = np.array([nucleotide, context, self.pick_cytosine_methylation(context)], dtype=float16)
                    contig_profile[f'{pos}_+_{insert_pos}'] = meth_profile
            else:
                for base in variant_info['iupac']:
                    if base == variant_info['reference']:
                        continue
                    else:
                        context = f'{ref_seq[pos - 1]}{base}{ref_seq[pos + 1]}'
                        if base == "C":
                            nucleotide = 0
                            context = 1 if ref_seq[pos] == "G" else 0
                        if base == "G":
                            nucleotide = 1
                            context = 1 if ref_seq[pos] == "C" else 0
                        
                        meth_profile = np.array([nucleotide, context, self.pick_cytosine_methylation(context)], dtype=float16)
                        contig_profile[f'{pos}_{variant_info["reference"]}_{base}'] = meth_profile